In [3]:
!pip install langchain_huggingface

  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)

  Attempting uninstall: huggingface-hub

    Found existing installation: huggingface-hub 0.30.2

    Uninstalling huggingface-hub-0.30.2:

      Successfully uninstalled huggingface-hub-0.30.2

   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
   ---------------------------------------- 0/2 [huggingface-hub]
  


[notice] A new release of pip is available: 25.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
!pip install tf-keras


[notice] A new release of pip is available: 25.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os

In [11]:
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

In [6]:
llm = ChatGroq(
    api_key=groq_api_key,  # Better to rename this variable to groq_api_key
    model="llama-3.3-70b-versatile",
    temperature=.7
)

# Without RAG

In [8]:
# Question not grounded in a document
question = "What is LangChain used for?"
print(llm.invoke(question))

content='LangChain is an open-source framework used for building applications that utilize large language models (LLMs). It provides a set of tools and libraries that enable developers to create a wide range of applications, from simple chatbots to complex AI-powered systems.\n\nLangChain is designed to simplify the process of working with LLMs, making it easier to integrate them into various applications. Some of the key use cases for LangChain include:\n\n1. **Conversational AI**: Building chatbots, voice assistants, and other conversational interfaces that can understand and respond to user input.\n2. **Text generation**: Generating text based on a given prompt or input, such as creating articles, stories, or product descriptions.\n3. **Language translation**: Translating text from one language to another using LLMs.\n4. **Summarization**: Summarizing long pieces of text into shorter, more digestible versions.\n5. **Question answering**: Building systems that can answer user questio

# Now with RAG

In [26]:
# Sample docs
docs = [
    "LangChain is an open-source framework that helps developers build applications powered by language models.",
    "It enables agents to interact with tools, memory, and external data sources.",
    "LangChain supports Retrieval-Augmented Generation to improve answer accuracy.",
    "LangChain's Latest version is Suffian,v828,289h"
]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_texts(docs, embeddings)
retriever = vectorstore.as_retriever()

In [27]:
prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context.
If you don't know the answer, say you don't know.

Context:
{context}

Question: {question}"""
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)



In [28]:
# Ask the same question again
response = qa_chain.invoke("What is LangChain used for?")
print(response)

LangChain is used to help developers build applications powered by language models.


In [29]:
response = qa_chain.invoke("LangChain's Latest version")
print(response)

Suffian, v828, 289h
